In [3]:
# Each sentence has words + the correct POS tag

train_data = [
    (
        ["The", "dog", "runs"],
        ["DET", "NOUN", "VERB"]
    ),
    (
        ["A", "cat", "sleeps"],
        ["DET", "NOUN", "VERB"]
    ),
    (
        ["The", "big", "dog", "runs"],
        ["DET", "ADJ", "NOUN", "VERB"]
    )
]

for words, tags in train_data:
    print("Words:", words)
    print("Tags :", tags)
    print()

Words: ['The', 'dog', 'runs']
Tags : ['DET', 'NOUN', 'VERB']

Words: ['A', 'cat', 'sleeps']
Tags : ['DET', 'NOUN', 'VERB']

Words: ['The', 'big', 'dog', 'runs']
Tags : ['DET', 'ADJ', 'NOUN', 'VERB']



In [4]:
def word_features(words, i):
    word = words[i]

    features = {
        "word=" + word.lower(): 1,
        "is_capitalized=" + str(word[0].isupper()): 1,
        "is_digit=" + str(word.isdigit()): 1,
        "prefix=" + word[:2].lower(): 1,
        "suffix=" + word[-2:].lower(): 1,
    }

    # Previous word
    if i > 0:
        features["prev_word=" + words[i-1].lower()] = 1
    else:
        features["BOS"] = 1

    # Next word
    if i < len(words) - 1:
        features["next_word=" + words[i+1].lower()] = 1
    else:
        features["EOS"] = 1

    return features

In [6]:
words = ["The", "dog", "runs"]

for i, word in enumerate(words):

    print(f"\nWORD: {word}")
    print(word_features(words, i))


WORD: The
{'word=the': 1, 'is_capitalized=True': 1, 'is_digit=False': 1, 'prefix=th': 1, 'suffix=he': 1, 'BOS': 1, 'next_word=dog': 1}

WORD: dog
{'word=dog': 1, 'is_capitalized=False': 1, 'is_digit=False': 1, 'prefix=do': 1, 'suffix=og': 1, 'prev_word=the': 1, 'next_word=runs': 1}

WORD: runs
{'word=runs': 1, 'is_capitalized=False': 1, 'is_digit=False': 1, 'prefix=ru': 1, 'suffix=ns': 1, 'prev_word=dog': 1, 'EOS': 1}


In [7]:
X_train = []
y_train = []

for words, tags in train_data:

    X_sentence = [
        word_features(words, i)
        for i in range(len(words))
    ]

    X_train.append(X_sentence)
    y_train.append(tags)

print("Features for first sentence:")
for word, features in zip(train_data[0][0], X_train[0]):
    print(word, "→", features)

Features for first sentence:
The → {'word=the': 1, 'is_capitalized=True': 1, 'is_digit=False': 1, 'prefix=th': 1, 'suffix=he': 1, 'BOS': 1, 'next_word=dog': 1}
dog → {'word=dog': 1, 'is_capitalized=False': 1, 'is_digit=False': 1, 'prefix=do': 1, 'suffix=og': 1, 'prev_word=the': 1, 'next_word=runs': 1}
runs → {'word=runs': 1, 'is_capitalized=False': 1, 'is_digit=False': 1, 'prefix=ru': 1, 'suffix=ns': 1, 'prev_word=dog': 1, 'EOS': 1}


In [9]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    max_iterations=100
)

crf.fit(X_train, y_train)

print("CRF trained!")

CRF trained!


In [11]:
print("STATE / WORD FEATURES\n")

for (feature, tag), weight in crf.state_features_.items():
    print(f"{feature:30s} → {tag:5s} : {weight:+.3f}")

STATE / WORD FEATURES

word=the                       → DET   : +0.281
is_capitalized=True            → DET   : +0.447
is_digit=False                 → DET   : +0.068
is_digit=False                 → NOUN  : +0.030
is_digit=False                 → VERB  : +0.041
is_digit=False                 → ADJ   : -0.140
prefix=th                      → DET   : +0.281
suffix=he                      → DET   : +0.281
BOS                            → DET   : +0.447
next_word=dog                  → DET   : +0.061
next_word=dog                  → ADJ   : +0.215
word=dog                       → NOUN  : +0.295
is_capitalized=False           → NOUN  : +0.191
is_capitalized=False           → VERB  : +0.184
is_capitalized=False           → ADJ   : +0.004
prefix=do                      → NOUN  : +0.295
suffix=og                      → NOUN  : +0.295
prev_word=the                  → NOUN  : +0.044
prev_word=the                  → ADJ   : +0.216
next_word=runs                 → NOUN  : +0.295
word=runs        

In [ ]:
    print("TAG TRANSITION FEATURES\n")

    for (previous_tag, current_tag), weight in crf.transition_features_.items():
        print(
            f"{previous_tag:5s} → {current_tag:5s} : {weight:+.3f}"
        )

TAG TRANSITION FEATURES

DET   → NOUN  : +0.352
DET   → ADJ   : +0.227
NOUN  → VERB  : +0.666
ADJ   → NOUN  : +0.223


In [13]:
test_words = ["The", "cat", "runs"]

X_test = [[
    word_features(test_words, i)
    for i in range(len(test_words))
]]

prediction = crf.predict(X_test)

print("Words:", test_words)
print("Predicted tags:", prediction[0])

Words: ['The', 'cat', 'runs']
Predicted tags: ['DET' 'NOUN' 'VERB']


The one picture I would explain to the student
WORDS→FEATURES→LEARNED WEIGHTS→SCORES→{
Forward → Z
Viterbi → Best tags
	​

Training
Prediction
	​

	​